# Attention Variants: MHA → GQA → MLA

Compare attention mechanisms by visualizing attention maps.
Uses the analysis toolkit to extract weights from the fused
attention path.

**Prerequisites:** `uv sync --extra dev --extra analysis`

In [ ]:
import mlx.core as mx

from lmxlab.analysis.attention import extract_attention_maps
from lmxlab.models.base import LanguageModel
from lmxlab.models.gpt import gpt_tiny
from lmxlab.models.llama import llama_tiny

## MHA: Multi-Head Attention (GPT)

Every head has its own Q, K, V projections.
All heads have independent key/value pairs.

In [ ]:
gpt_cfg = gpt_tiny()
gpt = LanguageModel(gpt_cfg)
mx.eval(gpt.parameters())

tokens = mx.array([[0, 1, 2, 3, 4, 5, 6, 7]], dtype=mx.int32)
maps = extract_attention_maps(gpt, tokens)

print(f"GPT: {gpt_cfg.block.n_heads} heads, MHA")
print(f"Attention maps: {list(maps.keys())}")
print(f"Shape per layer: {maps['layer_0'].shape}")
print(f"  = (batch, heads, seq, seq)")

In [ ]:
# Verify causal masking: position i can only attend to positions <= i
w = maps["layer_0"]
mx.eval(w)
print("Attention weights for head 0, layer 0:")
for i in range(8):
    row = [f"{w[0, 0, i, j].item():.2f}" for j in range(8)]
    print(f"  pos {i}: [{', '.join(row)}]")

## GQA: Grouped Query Attention (LLaMA)

Fewer KV heads than Q heads. Multiple Q heads share
the same K/V pair. Reduces KV cache memory.

In [ ]:
llama_cfg = llama_tiny()
llama = LanguageModel(llama_cfg)
mx.eval(llama.parameters())

maps_gqa = extract_attention_maps(llama, tokens)

n_q = llama_cfg.block.n_heads
n_kv = llama_cfg.block.effective_n_kv_heads
ratio = n_q // n_kv

print(f"LLaMA: {n_q} Q heads, {n_kv} KV heads (ratio {ratio}:1)")
print(f"Shape: {maps_gqa['layer_0'].shape}")
print(f"KV cache savings: {ratio}x smaller than MHA")

## Parameter Efficiency Comparison

GQA saves parameters in K/V projections while keeping
the same number of Q heads for expressiveness.

In [ ]:
d = 64  # d_model for tiny configs

# MHA: Q, K, V, O all d×d
mha_params = 4 * d * d

# GQA: Q and O are d×d, K and V are d×(d/ratio)
gqa_kv_dim = n_kv * (d // n_q)
gqa_params = 2 * d * d + 2 * d * gqa_kv_dim

savings = (1 - gqa_params / mha_params) * 100
print(f"MHA attention params: {mha_params:,}")
print(f"GQA attention params: {gqa_params:,}")
print(f"Savings: {savings:.0f}%")

## Visualize Attention (requires matplotlib)

In [ ]:
try:
    from lmxlab.analysis.plotting import plot_attention_heatmap

    fig = plot_attention_heatmap(maps["layer_0"], head=0, layer=0)
    fig.suptitle("GPT MHA — Layer 0, Head 0")
    fig.show()
except ImportError:
    print("Install matplotlib: uv sync --extra analysis")